# 05. Final Pipeline and Submission

This notebook documents how to reproduce the final selected submission from raw input parquet files. The end-to-end pipeline is implemented in `src/pipeline.py`; the notebook explains the steps and validates the existing CSV.

## Final selected submission

- File: `results/submit_iter07_prior_3seed.csv`
- Kaggle public NDCG@5: 0.39690
- Method: 3-seed LightGBM LambdaMART on 49 anchor features plus 12 leakage-safe historical priors over `prop_id`, `srch_destination_id`, and the pair `(prop_id, srch_destination_id)`. Score-averaged over seeds 42, 123, 456.
- Best iteration per seed (locked from the 80/10/10 split phase, written to `artifacts/seed_n_trees.json`): 291, 279, 375.

## Pipeline steps

`src/pipeline.py` does the following on full data:
1. Load `data/processed/train_clean.parquet` and `data/processed/test_clean.parquet`.
2. Build 49 anchor features (from `src/features.py`).
3. Build 12 leakage-safe historical priors (from `src/prior_features.py`):
   - For training rows: 5-fold group K-fold on `srch_id` with `fold_seed=42`, `alpha=20`, so each row gets priors aggregated over the OTHER 4 folds.
   - For test rows: full-train aggregates with the same Laplace smoothing.
4. Sort each matrix by `srch_id` so that LightGBM groups are contiguous.
5. For each seed in {42, 123, 456}, fit a LightGBM ranker on the FULL train slice with the best_iter values stored in `artifacts/seed_n_trees.json`. Predict on the test matrix.
6. Score-average across the three seeds.
7. Write `results/submit_final.csv` with header `srch_id,prop_id`, sorted by score within each `srch_id`.

## How to run from the command line

In [ ]:
# python src/pipeline.py --train --predict
# python src/pipeline.py --validate results/submit_iter07_prior_3seed.csv

## Validate the existing CSV

We do not retrain inside this notebook; we validate the file delivered with the project.

In [ ]:
from pathlib import Path
import sys
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.pipeline import validate_submission
info = validate_submission(ROOT / 'results' / 'submit_iter07_prior_3seed.csv')
info

## Negative results that did not enter the final pipeline

Listed here for completeness. None of these is in `submit_iter07_prior_3seed.csv`.

- A rank-level blend of the safe baseline, the diverse ensemble, and the historical-prior 3-seed reached holdout 0.401482 at its maximum, but was deliberately not selected to avoid the same-set selection pattern; the chosen 80/20 (final / diverse) blend reached 0.400453 holdout and was not uploaded.
- A fourth prior block keyed by `(prop_id, visitor_location_country_id)` with four extra features (block D) gave a same-seed delta of +0.000745 against the 3-seed prior baseline, below the +0.0010 acceptance threshold, and was rejected.

These are documented in our internal working notes; the report includes them only as honest negative results, not as production decisions.